# NB12 — Hotspots Top-15
**ZMM Movilidad Predictiva**

Identifica las 15 combinaciones temporal-ambientales con mayor riesgo predicho.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

RUTA_PROCESSED = '../data_processed/'
RUTA_OUTPUTS   = '../outputs/'

print('='*65)
print('NB12: HOTSPOTS TOP-15')
print('='*65)

In [ ]:
# Load data and model
df = pd.read_csv(RUTA_PROCESSED + 'super_tabla_con_clusters.csv', parse_dates=['fecha_hora'])
df = df[(df['fecha_hora'] >= '2023-01-01') & (df['fecha_hora'] <= '2025-12-31 23:00:00')].copy()

with open(RUTA_OUTPUTS + 'modelo_m2_rf.pkl', 'rb') as f:
    m2 = pickle.load(f)
model, features = m2['modelo'], m2['features']
print(f'Modelo: {m2["nombre"]} | Shape: {df.shape}')

# Recreate features
for sf in ['dist_industrial_promedio','dist_industrial_minima','siniestros_en_zona','masa_laboral_max']:
    df[sf] = df[sf].fillna(df[sf].median())
df['tiene_dato_espacial'] = 1
if 'tipo_dia' in df.columns:
    df = pd.concat([df, pd.get_dummies(df['tipo_dia'], prefix='td', drop_first=True)], axis=1)

In [ ]:
# Franjas
def franja(h):
    if 0<=h<6: return 'madrugada'
    elif 6<=h<9: return 'pico_manana'
    elif 9<=h<14: return 'mediodia'
    elif 14<=h<16: return 'cruce_turnos'
    elif 16<=h<19: return 'pico_tarde'
    else: return 'noche'

df['franja_horaria'] = df['hora_del_dia'].apply(franja)
df['tipo_dia_cat'] = np.where(df['es_fin_de_semana']==1, 'fin_de_semana', 'dia_laboral')
df['nivel_lluvia_cat'] = df['nivel_lluvia'].map({0:'seco',1:'brisa',2:'lluvia_leve',3:'lluvia_fuerte'}).fillna('seco')

# Predict
proba = model.predict_proba(df[features])
df['riesgo_predicho'] = proba[:,0]*0 + proba[:,1]*1 + proba[:,2]*2
df['combinacion'] = df['franja_horaria'] + '_' + df['tipo_dia_cat'] + '_' + df['nivel_lluvia_cat']
print(f'Combinaciones: {df["combinacion"].nunique()}')

In [ ]:
# Aggregate
hs = df.groupby('combinacion').agg(
    franja_horaria=('franja_horaria','first'), tipo_dia=('tipo_dia_cat','first'),
    nivel_lluvia=('nivel_lluvia_cat','first'), total_horas=('fecha_hora','count'),
    riesgo_promedio=('riesgo_predicho','mean'), riesgo_max=('riesgo_predicho','max'),
    siniestros_reales=('siniestros_zona_industrial','sum'),
    siniestros_promedio=('siniestros_zona_industrial','mean'),
    temperatura_promedio=('temperatura_c','mean')
).reset_index()

hs_f = hs[hs['total_horas'] >= 50].sort_values('riesgo_promedio', ascending=False)
top15 = hs_f.head(15).copy()
top15['rank'] = range(1, len(top15)+1)

franja_desc = {'madrugada':'Madrugada (00-06h)', 'pico_manana':'Pico Manana (06-09h)',
    'mediodia':'Mediodia (09-14h)', 'cruce_turnos':'Cruce Turnos (14-16h)',
    'pico_tarde':'Pico Tarde (16-19h)', 'noche':'Noche (19-23h)'}

print('\nTOP 15 HOTSPOTS:')
for _, r in top15.iterrows():
    desc = franja_desc.get(r['franja_horaria'], r['franja_horaria'])
    dia = 'Finde' if r['tipo_dia']=='fin_de_semana' else 'Laboral'
    print(f'  {r["rank"]:2d}. {desc} - {dia} - {r["nivel_lluvia"]}: riesgo={r["riesgo_promedio"]:.3f} ({r["total_horas"]} hrs)')

In [ ]:
# Save
def describir(r):
    p = [franja_desc.get(r['franja_horaria'],r['franja_horaria'])]
    p.append('Fin de semana' if r['tipo_dia']=='fin_de_semana' else 'Dia laboral')
    if r['nivel_lluvia'] != 'seco': p.append('Con '+r['nivel_lluvia'])
    return ' | '.join(p)

top15['descripcion'] = top15.apply(describir, axis=1)
top15['recomendacion'] = top15.apply(lambda r:
    'Priorizar patrullaje en {} los {}s'.format(r['franja_horaria'],r['tipo_dia'])
    if r['tipo_dia']=='dia_laboral'
    else 'Atencion especial: {} fines de semana'.format(r['franja_horaria']), axis=1)

top15.to_csv(RUTA_OUTPUTS + 'hotspots_top15.csv', index=False)
pres = top15[['rank','descripcion','riesgo_promedio','siniestros_reales','total_horas','recomendacion']].copy()
pres.columns = ['Rank','Perfil Temporal','Riesgo Predicho','Siniestros Historicos','Horas Observadas','Recomendacion']
pres.to_csv(RUTA_OUTPUTS + 'hotspots_presentacion.csv', index=False)
print('Saved: hotspots_top15.csv + hotspots_presentacion.csv')

In [ ]:
# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0,0].barh(range(len(top15)), top15['riesgo_promedio'], color='coral')
axes[0,0].set_yticks(range(len(top15)))
axes[0,0].set_yticklabels([f'{i+1}.{c[:22]}' for i,c in enumerate(top15['combinacion'])])
axes[0,0].set_xlabel('Riesgo Promedio'); axes[0,0].set_title('Top 15 Hotspots'); axes[0,0].invert_yaxis()

axes[0,1].barh(range(len(top15)), top15['siniestros_reales'], color='steelblue')
axes[0,1].set_yticks(range(len(top15)))
axes[0,1].set_yticklabels([f'{i+1}.{c[:22]}' for i,c in enumerate(top15['combinacion'])])
axes[0,1].set_xlabel('Siniestros Reales'); axes[0,1].set_title('Siniestros Historicos'); axes[0,1].invert_yaxis()

pivot_r = df.groupby(['franja_horaria','tipo_dia_cat'])['riesgo_predicho'].mean().unstack()
sns.heatmap(pivot_r, annot=True, fmt='.3f', cmap='Reds', ax=axes[1,0])
axes[1,0].set_title('Riesgo: Franja x Tipo Dia')

pivot_l = df.groupby(['franja_horaria','nivel_lluvia_cat'])['riesgo_predicho'].mean().unstack()
sns.heatmap(pivot_l, annot=True, fmt='.3f', cmap='Blues', ax=axes[1,1])
axes[1,1].set_title('Riesgo: Franja x Lluvia')

plt.tight_layout(); plt.savefig(RUTA_OUTPUTS + 'hotspots_visualizacion.png', dpi=150); plt.show()
print('NB12 COMPLETADO')